# GARNET Image Simulation Tutorial: Using image_generator.py

This notebook can be used to enerate simulated RSO images (in sets of `.fits`, `.png`, and labelme `.json` format data) from a TLE file and other optional information. 

image_generator.py is designed to function when run in terminal, however in this notebook it is imported as a module to allow users to call its `main()` funciton directly. You can change the variables when calling `main()` to see how passing different arguments will affect the resulting produced image(s).

As a note: `image_generator.py` imports its sibling modules directly, so the package folder must be on `sys.path`. The setup cell below shows an example of adding it before importing. Change `OUTPUT_DIR`, `TLE_FILE`, and `PKG_DIR` to match the file locations on your own computer before running.  

In [ ]:
# imports and file setup:

import os
import sys
from pathlib import Path
from glob import glob

# Paths to files: NOTE TO NOTEBOOK USERS: edit file paths these to match their locations on your own computer before running
OUTPUT_DIR = "C:/Users/Bruce Ritter/GISTDA/garnet_image_simulation/results_ignore/"
TLE_FILE   = "C:/Users/Bruce Ritter/GISTDA/garnet_image_simulation/resource/TLEs/NORAD_67683.txt"

# NOTE TO NOTEBOOK USERS: only edit part before "/"
PKG_DIR = Path("C:/Users/Bruce Ritter/GISTDA/garnet_image_simulation") / "garnet_image_simulation"

os.makedirs(OUTPUT_DIR, exist_ok=True)


if str(PKG_DIR) not in sys.path:
    sys.path.insert(0, str(PKG_DIR))

assert PKG_DIR.joinpath("image_generator.py").exists(), f"image_generator.py not found in {PKG_DIR}"
assert Path(TLE_FILE).exists(), f"TLE file not found: {TLE_FILE}"
print("Output dir:", OUTPUT_DIR)
print("TLE file:  ", TLE_FILE)
print("Package:   ", PKG_DIR)

In [ ]:
#importing image_generator.py:
import image_generator

In [ ]:
# if you edit image_generator.py, you can re-run this cell to pick up the changes
import importlib
importlib.reload(image_generator)
print("Imported image_generator from:", image_generator.__file__)

## Run the generator

The block below will call image_generator.py's `main()` directly. Observer/telescope params are left as `None`, so the script falls back to its built-in defaults (pixel size = 3.76 µm, focal length = 1050 mm, image size = 2128×3192 pix, binning = 2, ect.). With `exp_time = None`, the exposure time will be randomized between 0.5 - 5 seconds. With `rso_mag = None`, the magnitude of the RSO will be randomized between 5 and 12. Values below or above these stated ranges can still be accepted if input directly.

This configuration queries Gaia for background stars using the astroquery module to do an online query, so it needs an internet connection. Set `offline = True` to use the offline Gaia of Hipparchos Catalogs instead (Only if they are downloaded on your device, and in the case of non-gaiaoffline catalogs the catalog file must be specified with `cat_file`)

In [ ]:
image_generator.main(
    file_path=OUTPUT_DIR,
    TLE=TLE_FILE,
    n_generated=1,                   # Set to 1: will generate 1 set of images per mode per TLE in TLE_FILE
    mode_generated=None,             # None = both TRACKING and LEAPFROG; or 'TRACKING' / 'LEAPFROG'
    exp_time= None,                     # None = random exposure per pass
    when="2026-06-11 23:00:00.0000", # when = observing date/time. Leave blank to use current local time. 
    observer=None,                   # None / 'SKP' / 'DSC' / other
    obs_lat=None,                    # None = Matches SKP location
    obs_lon=None,                    # None = Matches SKP location 
    obs_alt=None,                    # None = Matches SKP location
    rso_mag=None,                    # None = random RSO magnitude
    mag_lim=16,                      # 16 = stars' limiting magnitude set to 16
    pix_size=None,                   # None = Matches current SKP telescope camera setting
    nrows=None,                      # None = Matches current SKP telescope camera setting
    ncols=None,                      # None = Matches current SKP telescope camera setting
    focal_len=None,                  # None = Matches current SKP telescope camera setting
    binning=2,                       # 2 = binning factor set to 2 (affects pixel scale)
    offline= False,                  # True = use offline Gaia catalog
    sky_vin=False,                   # False = vignette effect turned off
    bias=False,                      # False = bias noise turned off
    catalog="G",                     # "G" = local gaiaoffline downloaded catalog, defaults to trying to use a local gaiaoffline catalog is offline is specified
    cat_file=None                    # None = either local gaiaoffline or nothing (if online) 
)

print("Generation(s) Done.")



## Check the generated files

image_generator.py will add .fits, .png, and .json files to the same folder (folder named for observation/creation time, mode, NORAD ID, exposure time). 

The block of code below will display the first .png file generated in the output directory as a confirmation that files have been generated and saved in the correct location. To view .fits files, navigate to the output directory and open with your favorate .fits file viewing software. To view .json files, navigate to the output directory and open files. They should display in VScode. Or, if you use LabelMe and are using this tool for labeling purposes, input the json file and .fits or .png to LabelMe to display the image with labels overlaid. 

In [ ]:
search = os.path.join(os.path.dirname(OUTPUT_DIR), "**", "*.png")
pngs = sorted(glob(search, recursive=True))
print(f"Found {len(pngs)} PNG file(s):")
for p in pngs[:20]:
    print("  ", p)

if pngs:
    from PIL import Image
    display(Image.open(pngs[1]))

## Second Example: Using Offline Gaia Catalog


The cells below run un the same main image generator function but with offline = True, then run the same image viewer. This will display the most recent generated image in your specified output file directory. 

**This example requires an offline downloaded gaia catalog:** input configurations below use gaiaoffline functions to access an offline version of GAIA DR3 that can be downloaded in a number of ways, with instructions included in the ```gaia_offline_creator.ipynb``` example notebook.

In [ ]:
image_generator.main(
    file_path=OUTPUT_DIR,
    TLE=TLE_FILE,
    n_generated=1,                   # Set to 1: will generate 1 set of images per mode per TLE in TLE_FILE
    mode_generated=None,             # None = both TRACKING and LEAPFROG; or 'TRACKING' / 'LEAPFROG'
    exp_time= 1,                    # None = random exposure per pass
    when="2026-06-11 23:00:00.0000", # when = observing date/time. Leave blank to use current local time. 
    observer=None,                   # None / 'SKP' / 'DSC' / other
    obs_lat=None,                    # None = Matches SKP location
    obs_lon=None,                    # None = Matches SKP location 
    obs_alt=None,                    # None = Matches SKP location
    rso_mag=None,                    # None = random RSO magnitude
    mag_lim=16,                      # 16 = stars' limiting magnitude set to 16
    pix_size=None,                   # None = Matches current SKP telescope camera setting
    nrows=None,                      # None = Matches current SKP telescope camera setting
    ncols=None,                      # None = Matches current SKP telescope camera setting
    focal_len=None,                  # None = Matches current SKP telescope camera setting
    binning=2,                       # 2 = binning factor set to 2 (affects pixel scale)
    offline= True,                    # True = use offline Gaia catalog
    sky_vin=False,                   # False = vignette effect turned off
    bias=False,                      # False = bias noise turned off
    catalog="G",                     # "G" = defaults to trying to use a local gaiaoffline catalog is offline is specified , can be changed for different offline catalogs
    cat_file=None                    # None = no specific catalog file used: either online query will be used or if offline: local gaiaoffline catalog will be used
)
print("Done.")

In [ ]:
search = os.path.join(os.path.dirname(OUTPUT_DIR), "**", "*.png")
pngs = sorted(glob(search, recursive=True))
print(f"Found {len(pngs)} PNG file(s):")
for p in pngs[:20]:
    print("  ", p)

if pngs:
    from PIL import Image
    display(Image.open(pngs[0]))